# IzzyViz Use Case 1: Exploring RAG Attention Behavior

This notebook demonstrates the IzzyViz workflow on one RAG-style extractive QA example.

Workflow:

**Dataset Overview -> Attention Tensors -> Overview -> Select -> Cluster -> Inspect -> Explore**

Research focus:

- The base SQuAD-style context, question, and gold answer stay fixed.
- We add one retrieved context that contains the answer phrase and repeated answer-related evidence.
- We use IzzyViz to inspect whether attention heads focus on answer-bearing spans, repeated evidence, or structural tokens such as prompt prefixes and sinks.

The notebook generates the selected RAG example from the included dataset. It does not rely on pre-rendered local experiment images.

## 0. Clone and Install IzzyViz

This use case can be opened directly from GitHub/Colab. The setup cell below finds an existing local `IzzyViz` checkout or clones `https://github.com/zoeyada/IzzyViz.git`.

In local VS Code runs, the notebook uses the checkout directly on `sys.path`. In Colab, it installs IzzyViz in editable mode. The workflow itself computes attention tensors in memory, then runs Overview, Select, Cluster, and Inspect with the same API pattern as the quick-start workflow notebook.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/zoeyada/IzzyViz.git'
REPO_NAME = 'IzzyViz'

def find_or_clone_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'izzyviz').is_dir() and (candidate / 'workflow').is_dir():
            return candidate

    workspace = Path('/content') if Path('/content').exists() else cwd
    repo_dir = workspace / REPO_NAME
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', REPO_URL, str(repo_dir)])
    return repo_dir.resolve()

PROJECT_DIR = find_or_clone_project_dir()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if Path('/content').exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)])
else:
    print('Using local checkout on sys.path; skipping editable install.')

def ensure_numpy_pandas_abi():
    check_code = "import numpy, pandas; import numpy.random; print(numpy.__version__, pandas.__version__)"
    result = subprocess.run([sys.executable, '-c', check_code], text=True, capture_output=True)
    if result.returncode == 0:
        print('NumPy/Pandas:', result.stdout.strip())
        return

    print('Repairing NumPy/Pandas binary install:')
    print((result.stderr or result.stdout).strip().splitlines()[-1])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--force-reinstall',
        '--no-cache-dir',
        'numpy==1.26.4',
        'pandas==2.2.2',
    ])

ensure_numpy_pandas_abi()
print('Using IzzyViz project:', PROJECT_DIR)


Using local checkout on sys.path; skipping editable install.
NumPy/Pandas: 1.26.4 2.2.2
Using IzzyViz project: /home/cuizhouying/IzzyViz


## 1. Setup and Dataset Overview

This first overview is only about the RAG dataset content. It does not use model attention, clustering, or any computed visualization result.

Key parameters:

- `TARGET_CTX_ID`: the retrieved context to run through the full IzzyViz workflow.
- `ANSWER`: the gold SQuAD answer used as the analysis anchor.
- `MODEL_NAME`: the causal LM used to compute attention tensors for this RAG prompt.


In [2]:
from pathlib import Path
import gc
import json
import os
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, IFrame, Markdown
from transformers import AutoModelForCausalLM, AutoTokenizer

from izzyviz import (
    visualize_attention_overview_fast,
    select_attention_heads_by_metric,
    cluster_attention_heads,
    visualize_attention_matrix,
)

PROJECT_DIR = Path(PROJECT_DIR) if 'PROJECT_DIR' in globals() else Path.cwd()
if PROJECT_DIR.name != 'IzzyViz' and (PROJECT_DIR.parent / 'izzyviz').exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_DIR = PROJECT_DIR / 'workflow' / 'rag_cluster' / 'rag_contexts_answer_bearing'
OUTPUT_ROOT = DATASET_DIR / 'exp_function'
CONTEXTS_PATH = DATASET_DIR / 'external_contexts.json'

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
SELECTED_CTX_IDS = ['ctx_ac_001', 'ctx_0039', 'ctx_0088']
ANSWER = 'Lobund Institute for Animal Studies'
ANSWER_WORDS = ['lobund', 'institute', 'animal', 'studies']
SAVE_DATA_OUTPUTS = True
HF_LOCAL_FILES_ONLY = False
TOP_K = 3
N_CLUSTERS = 3
COMPARISON_OUTPUT_DIR = OUTPUT_ROOT / 'rag_three_context_comparison_pdf'
COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CONTEXTS_PATH.exists():
    raise FileNotFoundError(f'Missing RAG dataset: {CONTEXTS_PATH}')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def count_answer_word_hits(text):
    text_l = text.lower()
    return sum(text_l.count(word) for word in ANSWER_WORDS)

def safe_name(value):
    import re
    value = str(value)
    value = re.sub(r'[^a-zA-Z0-9_\-]+', '_', value)
    return value.strip('_') or 'NA'

def show_pdf(path, width=1050, height=720):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    display(Markdown(f'`{path.name}`'))
    display(IFrame(src=str(path), width=width, height=height))

contexts = load_json(CONTEXTS_PATH)
missing = [ctx_id for ctx_id in SELECTED_CTX_IDS if ctx_id not in contexts]
if missing:
    raise KeyError(f'Missing selected contexts: {missing}')

dataset_rows = []
for ctx_id, item in contexts.items():
    answer_overlap = item.get('answer_overlap') or {}
    text = item.get('text', '')
    dataset_rows.append({
        'ctx_id': ctx_id,
        'length': int(item.get('length', -1)),
        'relatedness': item.get('relatedness', 'unknown'),
        'full_answer_present': bool(answer_overlap.get('full_answer_present', False)),
        'answer_overlap_ratio': float(answer_overlap.get('overlap_ratio', 0.0) or 0.0),
        'answer_word_hits': count_answer_word_hits(text),
        'selected': ctx_id in SELECTED_CTX_IDS,
        'text_preview': text[:260].replace('\n', ' '),
    })

dataset_df = pd.DataFrame(dataset_rows).sort_values(['relatedness', 'full_answer_present', 'length', 'ctx_id']).reset_index(drop=True)
selected_df = dataset_df[dataset_df['ctx_id'].isin(SELECTED_CTX_IDS)].copy()
selected_df['selection_order'] = selected_df['ctx_id'].map({ctx_id: i + 1 for i, ctx_id in enumerate(SELECTED_CTX_IDS)})
selected_df = selected_df.sort_values('selection_order').reset_index(drop=True)

print('Project:', PROJECT_DIR)
print('Dataset:', DATASET_DIR)
print('Output PDF folder:', COMPARISON_OUTPUT_DIR)
print('Number of retrieved contexts:', len(dataset_df))
print('Selected contexts:', ', '.join(SELECTED_CTX_IDS))
print('Metric top-k:', TOP_K)
print('Cluster count:', N_CLUSTERS)

display(selected_df[['selection_order', 'ctx_id', 'relatedness', 'length', 'full_answer_present', 'answer_overlap_ratio', 'answer_word_hits', 'text_preview']])


Project: /home/cuizhouying/IzzyViz
Dataset: /home/cuizhouying/IzzyViz/workflow/rag_cluster/rag_contexts_answer_bearing
Output PDF folder: /home/cuizhouying/IzzyViz/workflow/rag_cluster/rag_contexts_answer_bearing/exp_function/rag_three_context_comparison_pdf
Number of retrieved contexts: 58
Selected contexts: ctx_ac_001, ctx_0039, ctx_0088
Metric top-k: 3
Cluster count: 3


,selection_order,ctx_id,relatedness,length,full_answer_present,answer_overlap_ratio,answer_word_hits,text_preview
0,1,ctx_ac_001,high,96,True,1.00,6,mission and an expanded student body. He stres...
1,2,ctx_0039,medium,1801,False,0.50,47,"Animal testing, also known as animal experimen..."
2,3,ctx_0088,low,2049,False,0.25,0,The Eiffel Tower ( EYE-fəl; French: Tour Eiffe...


In [3]:
related_order = ['high', 'medium', 'low']
color_map = {'high': '#2b8cbe', 'medium': '#f0a202', 'low': '#6a994e', 'unknown': '#777777'}
selected_colors = [color_map.get(v, '#777777') for v in selected_df['relatedness']]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

related_counts = dataset_df['relatedness'].value_counts().reindex(related_order, fill_value=0)
axes[0].bar(related_counts.index.astype(str), related_counts.values, color=[color_map[k] for k in related_counts.index])
axes[0].set_title('Retrieved Contexts by Relatedness')
axes[0].set_xlabel('Relatedness')
axes[0].set_ylabel('Count')

for relatedness, group in dataset_df.groupby('relatedness'):
    axes[1].scatter(
        group['length'],
        group['answer_word_hits'],
        s=36,
        alpha=0.55,
        color=color_map.get(relatedness, '#777777'),
        label=relatedness,
    )
axes[1].scatter(
    selected_df['length'],
    selected_df['answer_word_hits'],
    s=180,
    facecolors='none',
    edgecolors='#d62728',
    linewidths=2.2,
    label='selected',
)
for _, row in selected_df.iterrows():
    axes[1].annotate(
        f"{int(row['selection_order'])}. {row['ctx_id']}",
        (row['length'], row['answer_word_hits']),
        textcoords='offset points',
        xytext=(8, 8),
        fontsize=9,
        color='#7a1f1f',
    )
axes[1].set_title('Selected Diverse Contexts')
axes[1].set_xlabel('Context length')
axes[1].set_ylabel('Answer word hits')
axes[1].legend(fontsize=8)

x = np.arange(len(selected_df))
axes[2].bar(x - 0.18, selected_df['length'], width=0.36, color=selected_colors, label='length')
axes[2].set_ylabel('Length')
axes[2].set_title('Three-Context Comparison Inputs')
axes[2].set_xticks(x)
axes[2].set_xticklabels(selected_df['ctx_id'], rotation=20, ha='right')
ax2 = axes[2].twinx()
ax2.bar(x + 0.18, selected_df['answer_overlap_ratio'], width=0.36, color='#444444', alpha=0.45, label='answer overlap')
ax2.set_ylim(0, 1.05)
ax2.set_ylabel('Answer overlap ratio')

handles1, labels1 = axes[2].get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
axes[2].legend(handles1 + handles2, labels1 + labels2, fontsize=8, loc='upper right')

dataset_overview_path = COMPARISON_OUTPUT_DIR / '00_dataset_overview_selected_three_contexts.pdf'
fig.savefig(dataset_overview_path, bbox_inches='tight')
plt.show()
show_pdf(dataset_overview_path, height=520)

for _, row in selected_df.iterrows():
    print(f"\n[{int(row['selection_order'])}] {row['ctx_id']} | {row['relatedness']} | length={row['length']} | full_answer={row['full_answer_present']} | hits={row['answer_word_hits']}")
    print(contexts[row['ctx_id']].get('text', '')[:700])



[1] ctx_ac_001 | high | length=96 | full_answer=True | hits=6
mission and an expanded student body. He stressed advanced studies and research while quadrupling the university's student population, with undergraduate enrollment increasing by more than half and graduate student enrollment growing fivefold. Cavanaugh established the Lobund Institute for Animal Studies and Notre Dame's Medieval Institute, presided over the construction of Nieuwland Science Hall, Fisher Hall, and the Morris Inn, and the Hall of Liberal Arts (now O'Shaughnessy Hall), made

[2] ctx_0039 | medium | length=1801 | full_answer=False | hits=47
Animal testing, also known as animal experimentation, animal research, and in vivo testing, is the use of non-human animals, as model organisms, in experiments that seek answers to scientific and medical questions. This approach can be contrasted with field studies in which animals are observed in their natural environments or habitats. Experimental research with animals is

`00_dataset_overview_selected_three_contexts.pdf`

## 2. Attention Tensors

This section follows the quick-start workflow notebook: build the RAG prompt, run one forward pass, keep `attentions` in memory, and define the `x` and `y` slices used by every visualization stage.


In [4]:
BASE_CONTEXT = (
    "The Rev. John J. Cavanaugh, C.S.C. served as president from 1946 to 1952. "
    "Cavanaugh's legacy at Notre Dame in the post-war years was devoted to raising academic standards "
    "and reshaping the university administration to suit it to an enlarged educational mission and an expanded "
    "student body and stressing advanced studies and research at a time when Notre Dame quadrupled in student census, "
    "undergraduate enrollment increased by more than half, and graduate student enrollment grew fivefold. "
    "Cavanaugh also established the Lobund Institute for Animal Studies and Notre Dame's Medieval Institute. "
    "Cavanaugh also presided over the construction of the Nieuwland Science Hall, Fisher Hall, and the Morris Inn, "
    "as well as the Hall of Liberal Arts (now O'Shaughnessy Hall), made possible by a donation from I.A. O'Shaughnessy, "
    "at the time the largest ever made to an American Catholic university. "
    "Cavanaugh also established a system of advisory councils at the university, which continue today and are vital "
    "to the university's governance and development."
)
QUESTION = 'Which institute involving animal life did Cavanaugh create at Notre Dame?'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

print('Loading tokenizer:', MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False,
    local_files_only=HF_LOCAL_FILES_ONLY,
)

print('Loading model:', MODEL_NAME)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    attn_implementation='eager',
    trust_remote_code=False,
    local_files_only=HF_LOCAL_FILES_ONLY,
)
model.to(DEVICE)
model.eval()
print('Loaded in', round(time.time() - t0, 1), 's')
print('Device:', DEVICE)
print('Model dtype:', next(model.parameters()).dtype)

def build_prompt(ctx_item):
    rag_context_text = f"RAG Context: {ctx_item['text']}\n"
    base_context_text = f'Context: {BASE_CONTEXT}\n'
    question_text = f'Question: {QUESTION}\n'
    answer_prefix = 'Answer:'
    answer_text = f' {ANSWER}'
    x_text = rag_context_text + base_context_text + question_text
    prompt = x_text + answer_prefix + answer_text
    return prompt, x_text, answer_prefix


Loading tokenizer: Qwen/Qwen2.5-1.5B
Loading model: Qwen/Qwen2.5-1.5B
Loaded in 0.7 s
Device: cpu
Model dtype: torch.float32


/home/cuizhouying/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|##########| 338/338 [00:00<00:00, 1450.25it/s]


## 3. Three-Context Comparison Workflow


In [5]:
METRIC_CONFIGS = [
    {
        'label': 'entropy',
        'metric_name': 'entropy',
        'metric_params': {},
        'title': 'Entropy',
    },
    {
        'label': 'variance',
        'metric_name': 'variance',
        'metric_params': {},
        'title': 'Variance',
    },
    {
        'label': 'threshold_based_concentration',
        'metric_name': 'threshold_mass',
        'metric_params': {'threshold': 0.05},
        'title': 'Threshold-based concentration',
    },
    {
        'label': 'top_percent_concentration',
        'metric_name': 'top_percent_mass',
        'metric_params': {'top_percent': 0.05},
        'title': 'Top-percent concentration',
    },
    {
        'label': 'local_dependency',
        'metric_name': 'low_range',
        'metric_params': {'max_distance': 5},
        'title': 'Local dependency',
    },
    {
        'label': 'long_range_dependency',
        'metric_name': 'long_range',
        'metric_params': {'min_distance': 20},
        'title': 'Long-range dependency',
    },
]
comparison_summaries = []

for ctx_index, ctx_id in enumerate(SELECTED_CTX_IDS, start=1):
    ctx_item = contexts[ctx_id]
    relatedness = safe_name(ctx_item.get('relatedness', 'NA'))
    prefix = f'{ctx_index:02d}_{safe_name(ctx_id)}_{relatedness}_'
    display(Markdown(f'## Context {ctx_index}: `{ctx_id}` ({ctx_item.get("relatedness")}, length={ctx_item.get("length")})'))

    prompt, x_text, answer_prefix = build_prompt(ctx_item)
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(DEVICE)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    x_end = len(tokenizer(x_text, add_special_tokens=False)['input_ids'])
    answer_start = len(tokenizer(x_text + answer_prefix, add_special_tokens=False)['input_ids'])
    answer_end = len(tokens)
    x_labels = tokens[:x_end]
    y_labels = tokens[answer_start:answer_end]

    print('Sequence length:', len(tokens))
    print('x span = RAG context + base context + question:', (0, x_end))
    print('y span = answer:', (answer_start, answer_end))
    print('Answer tokens:', tokens[answer_start:answer_end])

    print('Running model forward pass...')
    t0 = time.time()
    with torch.no_grad():
        outputs = model(
            **inputs,
            output_attentions=True,
            return_dict=True,
            use_cache=False,
        )
    print('Forward done in', round(time.time() - t0, 1), 's')
    attentions = outputs.attentions
    attention_region = np.stack([
        layer_attn[0, :, answer_start:answer_end, 0:x_end].detach().float().cpu().numpy()
        for layer_attn in attentions
    ])
    print('answer -> RAG/context/question region:', attention_region.shape)

    overview_no_merge_path = COMPARISON_OUTPUT_DIR / f'{prefix}01_overview_no_merge_answer_to_rag_context_question.pdf'
    visualize_attention_overview_fast(
        attentions,
        query_slice=(answer_start, answer_end),
        key_slice=(0, x_end),
        title=f'{ctx_id}: answer -> RAG context/base context/question attention',
        save_path=overview_no_merge_path,
        shared_color_scale=True,
        shared_cbar=True,
        shared_cbar_label='Attention score',
        max_display_cols=96,
        figsize=(34, 58),
        wspace=0.26,
        hspace=0.30,
        close_after_save=True,
    )
    show_pdf(overview_no_merge_path)

    overview_merge_path = COMPARISON_OUTPUT_DIR / f'{prefix}02_overview_merge_tokens_answer_to_rag_context_question.pdf'
    visualize_attention_overview_fast(
        attentions,
        query_slice=(answer_start, answer_end),
        key_slice=(0, x_end),
        title=f'{ctx_id}: answer -> RAG context/base context/question attention (merged virtual tokens)',
        save_path=overview_merge_path,
        shared_color_scale=True,
        shared_cbar=True,
        shared_cbar_label='Attention score',
        merge_virtual_tokens=True,
        overview_top_n=3,
        show_merge_token_labels=True,
        merge_token_label_mode='index',
        merge_token_important_label_mode='index',
        merge_token_label_fontsize=3,
        x_labels=x_labels,
        y_labels=y_labels,
        figsize=(34, 58),
        wspace=0.34,
        hspace=0.38,
        close_after_save=True,
    )
    show_pdf(overview_merge_path)

    metric_results_by_label = {}
    primary_selection_records = None

    for metric_config in METRIC_CONFIGS:
        metric_label = metric_config['label']
        metric_name = metric_config['metric_name']
        metric_params = metric_config['metric_params']
        metric_title = metric_config['title']

        selection_result = select_attention_heads_by_metric(
            attentions,
            query_slice=(answer_start, answer_end),
            key_slice=(0, x_end),
            tokens=tokens,
            metric_name=metric_name,
            metric_params=metric_params,
            ignore_first_n=1,
            top_k=TOP_K,
            output_dir=None,
            run_name=f'{ctx_id}_{metric_label}_answer_to_rag_x',
            plot_overview=True,
            plot_overview_no_merge=True,
            plot_overview_merge=True,
            plot_importance_heatmap=True,
            plot_top_attention_maps=True,
            top_attention_count=TOP_K,
            overview_renderer='fast',
            overview_top_n=3,
            overview_kwargs={
                'figsize': (34, 58),
                'shared_cbar': True,
                'shared_cbar_label': 'Attention score',
                'highlight_top_cells': True,
                'highlight_ranked_only_no_merge': True,
                'highlight_ranked_only_merge': False,
                'show_merge_token_labels': True,
                'merge_token_label_mode': 'index',
                'merge_token_important_label_mode': 'index',
                'merge_token_label_fontsize': 3,
                'wspace': 0.34,
                'hspace': 0.38,
            },
            detail_top_n=20,
            detail_merge_virtual_tokens=True,
            detail_file_format='pdf',
            detail_kwargs={
                'xlabel': 'x: RAG Context + Base Context + Question',
                'ylabel': 'y: Answer',
                'length_threshold': 64,
                'if_interval': False,
                'if_top_cells': True,
                'show_scores_in_enlarged_cells': False,
                'lean_more': True,
            },
            close_after_save=False,
            save_summary=False,
        )

        selection_df = pd.DataFrame(selection_result['top_records'])
        display(Markdown(f'### Metric-based `{metric_title}` top {TOP_K} heads'))
        display(selection_df)

        selection_paths = {}
        for figure_name, fig in selection_result.get('figures', {}).items():
            pdf_path = COMPARISON_OUTPUT_DIR / f'{prefix}03_select_{metric_label}_{figure_name}.pdf'
            fig.savefig(pdf_path, bbox_inches='tight')
            plt.close(fig)
            selection_paths[figure_name] = str(pdf_path)

        for figure_name in ['overview_no_merge', 'overview_merge', 'importance_heatmap', 'rank_01_attention', 'rank_02_attention', 'rank_03_attention']:
            if figure_name in selection_paths:
                show_pdf(selection_paths[figure_name], width=1050, height=720 if 'overview' in figure_name else 640)

        metric_results_by_label[metric_label] = {
            'metric_name': metric_name,
            'metric_params': metric_params,
            'top_records': selection_result['top_records'],
            'output_paths': selection_paths,
        }
        if metric_label == 'long_range_dependency':
            primary_selection_records = selection_result['top_records']

    if primary_selection_records is None:
        primary_selection_records = next(iter(metric_results_by_label.values()))['top_records']

    cluster_result = cluster_attention_heads(
        attentions,
        query_slice=(answer_start, answer_end),
        key_slice=(0, x_end),
        tokens=tokens,
        metrics=[
            'concentration',
            'entropy',
            'variance',
            'threshold_mass',
            'top_percent_mass',
            'low_range',
            'long_range',
        ],
        metric_params={
            'threshold_mass': {'threshold': 0.05},
            'top_percent_mass': {'top_percent': 0.05},
            'low_range': {'max_distance': 5},
            'long_range': {'min_distance': 20},
        },
        n_clusters=N_CLUSTERS,
        representative_per_cluster=1,
        ignore_first_n=1,
        max_detail_plots=N_CLUSTERS,
        output_dir=COMPARISON_OUTPUT_DIR,
        output_prefix=f'{prefix}04_cluster_',
        overview_file_format='pdf',
        pca_file_format='pdf',
        detail_file_format='pdf',
        run_name=f'{ctx_id}_answer_to_rag_x_cluster',
        overview_renderer='fast',
        plot_overview=True,
        plot_overview_no_merge=True,
        plot_overview_merge=True,
        plot_pca=True,
        plot_detail_heatmaps=True,
        overview_top_n=5,
        overview_kwargs={
            'figsize': (34, 58),
            'shared_cbar': True,
            'shared_cbar_label': 'Attention score',
            'show_merge_token_labels': True,
            'merge_token_labels_representatives_only': False,
            'merge_token_label_mode': 'index',
            'merge_token_important_label_mode': 'index',
            'merge_token_label_fontsize': 3,
            'wspace': 0.34,
            'hspace': 0.38,
        },
        detail_top_n=20,
        detail_merge_virtual_tokens=True,
        save_summary=SAVE_DATA_OUTPUTS,
        detail_kwargs={
            'xlabel': 'x: RAG Context + Base Context + Question',
            'ylabel': 'y: Answer',
            'length_threshold': 64,
            'if_interval': False,
            'if_top_cells': True,
            'show_scores_in_enlarged_cells': False,
            'lean_more': True,
        },
    )

    cluster_sizes = pd.Series(cluster_result['labels']).value_counts().sort_index().to_dict()
    representative_records = []
    for idx in cluster_result['representatives']:
        layer, head = cluster_result['head_infos'][idx]
        cluster_id = int(cluster_result['labels'][idx])
        representative_records.append({
            'ctx_id': ctx_id,
            'cluster': cluster_id,
            'layer': int(layer),
            'head': int(head),
            'cluster_size': int(cluster_sizes.get(cluster_id, 0)),
        })
        print(f'cluster {cluster_id}: L{layer:02d} H{head:02d}')

    representative_df = pd.DataFrame(representative_records).sort_values('cluster').reset_index(drop=True)
    display(Markdown(f'### Clustering representatives ({N_CLUSTERS} clusters)'))
    display(representative_df)

    for key in ['pca', 'overview_no_merge', 'overview_merge']:
        if key in cluster_result['output_paths']:
            show_pdf(cluster_result['output_paths'][key], width=1050, height=720)

    for detail_path in cluster_result['output_paths'].get('detail_heatmaps', []):
        show_pdf(detail_path, width=1050, height=640)

    if representative_records:
        inspect_layer = representative_records[0]['layer']
        inspect_head = representative_records[0]['head']
    else:
        inspect_layer = primary_selection_records[0]['layer']
        inspect_head = primary_selection_records[0]['head']

    inspect_path = COMPARISON_OUTPUT_DIR / f'{prefix}05_inspect_attention_matrix_L{inspect_layer}_H{inspect_head}.pdf'
    visualize_attention_matrix(
        attention_region[inspect_layer, inspect_head],
        x_labels=x_labels,
        y_labels=y_labels,
        title=f'Inspect RAG Attention: {ctx_id} L{inspect_layer} H{inspect_head}',
        xlabel='x: RAG Context + Base Context + Question',
        ylabel='y: Answer',
        top_n=20,
        enlarged_size=2.0,
        gamma=1.5,
        cmap='Purples',
        merge_virtual_tokens=True,
        virtual_token_min_run=2,
        length_threshold=64,
        if_interval=False,
        if_top_cells=True,
        show_scores_in_enlarged_cells=False,
        lean_more=True,
        save_path=inspect_path,
        close_after_save=True,
    )
    display(Markdown(f'### Inspect: L{inspect_layer} H{inspect_head}'))
    show_pdf(inspect_path, width=1050, height=640)

    comparison_summaries.append({
        'ctx_id': ctx_id,
        'relatedness': ctx_item.get('relatedness'),
        'length': ctx_item.get('length'),
        'sequence_length': len(tokens),
        'x_end': x_end,
        'answer_start': answer_start,
        'answer_end': answer_end,
        'metric_top3_by_metric': {
            label: result['top_records']
            for label, result in metric_results_by_label.items()
        },
        'cluster_representatives': representative_records,
        'inspect_layer': int(inspect_layer),
        'inspect_head': int(inspect_head),
    })

    del outputs, attentions, attention_region, inputs, tokens
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

summary_path = COMPARISON_OUTPUT_DIR / '99_three_context_comparison_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(comparison_summaries, f, indent=2)

display(Markdown('## Comparison Summary'))
display(pd.DataFrame([
    {
        'ctx_id': item['ctx_id'],
        'relatedness': item['relatedness'],
        'length': item['length'],
        'sequence_length': item['sequence_length'],
        'inspect_layer': item['inspect_layer'],
        'inspect_head': item['inspect_head'],
    }
    for item in comparison_summaries
]))
print('Saved PDF folder:', COMPARISON_OUTPUT_DIR)
print('Saved summary:', summary_path)


Sequence length: 342
x span = RAG context + base context + question: (0, 334)
y span = answer: (336, 342)
Answer tokens: ['ĠLob', 'und', 'ĠInstitute', 'Ġfor', 'ĠAnimal', 'ĠStudies']
Running model forward pass...
Forward done in 1.8 s
answer -> RAG/context/question region: (28, 12, 6, 334)
cluster 0: L22 H07
cluster 1: L11 H01
cluster 2: L09 H00
Sequence length: 2110
x span = RAG context + base context + question: (0, 2102)
y span = answer: (2104, 2110)
Answer tokens: ['ĠLob', 'und', 'ĠInstitute', 'Ġfor', 'ĠAnimal', 'ĠStudies']
Running model forward pass...
Forward done in 12.7 s
answer -> RAG/context/question region: (28, 12, 6, 2102)
cluster 0: L09 H05
cluster 1: L23 H08
cluster 2: L14 H04
Sequence length: 2409
x span = RAG context + base context + question: (0, 2401)
y span = answer: (2403, 2409)
Answer tokens: ['ĠLob', 'und', 'ĠInstitute', 'Ġfor', 'ĠAnimal', 'ĠStudies']
Running model forward pass...
Forward done in 14.7 s
answer -> RAG/context/question region: (28, 12, 6, 2401)
clus

## Context 1: `ctx_ac_001` (high, length=96)

`01_ctx_ac_001_high_01_overview_no_merge_answer_to_rag_context_question.pdf`

`01_ctx_ac_001_high_02_overview_merge_tokens_answer_to_rag_context_question.pdf`

### Metric-based `Entropy` top 3 heads

...

`03_ctx_0088_low_04_cluster_cluster_2_L9_H5.pdf`

### Inspect: L16 H4

`03_ctx_0088_low_05_inspect_attention_matrix_L16_H4.pdf`

## Comparison Summary

,ctx_id,relatedness,length,sequence_length,inspect_layer,inspect_head
0,ctx_ac_001,high,96,342,22,7
1,ctx_0039,medium,1801,2110,9,5
2,ctx_0088,low,2049,2409,16,4


## 4. Generated PDF Outputs


In [6]:
pdf_paths = sorted(COMPARISON_OUTPUT_DIR.glob('*.pdf'))
print(f'PDF count: {len(pdf_paths)}')
for path in pdf_paths:
    print(path.name)


PDF count: 154
00_dataset_overview_selected_three_contexts.pdf
01_ctx_ac_001_high_01_overview_no_merge_answer_to_rag_context_question.pdf
01_ctx_ac_001_high_02_overview_merge_tokens_answer_to_rag_context_question.pdf
01_ctx_ac_001_high_03_select_entropy_importance_heatmap.pdf
01_ctx_ac_001_high_03_select_entropy_overview_merge.pdf
01_ctx_ac_001_high_03_select_entropy_overview_no_merge.pdf
01_ctx_ac_001_high_03_select_entropy_rank_01_attention.pdf
01_ctx_ac_001_high_03_select_entropy_rank_02_attention.pdf
01_ctx_ac_001_high_03_select_entropy_rank_03_attention.pdf
01_ctx_ac_001_high_03_select_local_dependency_importance_heatmap.pdf
01_ctx_ac_001_high_03_select_local_dependency_overview_merge.pdf
01_ctx_ac_001_high_03_select_local_dependency_overview_no_merge.pdf
01_ctx_ac_001_high_03_select_local_dependency_rank_01_attention.pdf
01_ctx_ac_001_high_03_select_local_dependency_rank_02_attention.pdf
01_ctx_ac_001_high_03_select_local_dependency_rank_03_attention.pdf
01_ctx_ac_001_high_03_sel

## 5. Notes


In [7]:
print('This notebook now compares three diverse RAG contexts end-to-end:')
print('- two overview PDFs per context: no merge and merged virtual tokens')
print('- metric-based selection with top 3 heads for entropy, variance, threshold-based concentration, top-percent concentration, local dependency, and long-range dependency')
print('- clustering with 3 clusters, PCA, overview labels, and representative heatmaps')
print('- one inspect attention matrix per context using visualize_attention_matrix')
print('All generated figures are saved as PDFs in:', COMPARISON_OUTPUT_DIR)


This notebook now compares three diverse RAG contexts end-to-end:
- two overview PDFs per context: no merge and merged virtual tokens
- metric-based selection with top 3 heads for entropy, variance, threshold-based concentration, top-percent concentration, local dependency, and long-range dependency
- clustering with 3 clusters, PCA, overview labels, and representative heatmaps
- one inspect attention matrix per context using visualize_attention_matrix
All generated figures are saved as PDFs in: /home/cuizhouying/IzzyViz/workflow/rag_cluster/rag_contexts_answer_bearing/exp_function/rag_three_context_comparison_pdf
